# Decomposition of FID into Sinusoidal Signals

The decomposition of an experimental FID into a superposition of sinusoidal signals
would deliver very useful information on frequencies, phases, amplitudes, and decay rates.
Such a list of parameters could then be compared with a database of expected signals, and
be used directly for quantification. This idea has been explored from the standpoint of
Bayesian probability theory by Bretthorst (https://www.sciencedirect.com/science/article/pii/002223649090287J}[J. Magn. Reson. 1990 ]), and applied by Krishnamurthy (https://onlinelibrary.wiley.com/doi/abs/10.1002/mrc.4022) to quantification, in combination with division of the FID into sub-FIDs corresponding 
to regions of interest in the spectrum.

Here, we explore the idea again and implement its basics in Julia.

In [ ]:
import Pkg
Pkg.activate(".")

using Revise
using Plots
using NMRlab
using JLD2

theme(:solarized_light)
default(size=(600,400),
        tick_direction=:out,
        framestyle=:box,
        margin=(6,:mm),
        legendfontsize=8,
        guidefontsize=10,
        tickfontsize=10,
        fontfamily="Helvetica", 
        fmt=:svg,dpi=100)

In [ ]:
@load "processedData.jld2"

In [ ]:
Plots.plot(NMRlab.coords(fidData,1),real(fidData[:,1]),
    xlabel="time (s)",
    ylabel="Amplitude",
    xlims=(0,0.1))

Here is a function that will superpose $n$ oscillators. The amplitudes, frequencies, phases, and decay rates are provided as a linear parameter array `B`.

In [ ]:
# B layout: [re(c₁), im(c₁), freq₁, γ₁,  re(c₂), ...]
# decay rate = DECAY_MAX * sigmoid(γ), bounded in (0, DECAY_MAX)
const DECAY_MAX = 20π   # ~10 Hz maximum linewidth

function oscillators2(r, B)
    l = length(B)
    n = Int(l/4)
    cr    = B[1:4:l]
    ci    = B[2:4:l]
    freq  = 1000*B[3:4:l]
    decay = DECAY_MAX ./ (1 .+ exp.(-B[4:4:l]))
    return sum((cr[k] + im*ci[k]) .* exp.((-decay[k] + im*freq[k]) .* r) for k in 1:n)
end


In order to fit experimental data with a set of oscillators, we need
the Jacobian $$J(t,\mathbf{B}) = \nabla_\mathbf{B} F(t;\mathbf{B})$$.

In [ ]:
function oscillators2_jacobian(r, B)
    l  = length(B)
    n  = Int(l/4)
    nr = length(r)

    cr    = B[1:4:l]
    ci    = B[2:4:l]
    freq  = 1000*B[3:4:l]
    decay = DECAY_MAX ./ (1 .+ exp.(-B[4:4:l]))   # sigmoid reparametrisation

    J = zeros(2*nr, l)

    for k in 1:n
        c  = cr[k] + im*ci[k]
        e  = exp.((-decay[k] + im*freq[k]) .* r)
        ce = c .* e

        i_cr, i_ci, i_freq, i_decay = 4k-3, 4k-2, 4k-1, 4k

        # ∂/∂re(c_k) = e
        J[1:nr,     i_cr] =  real(e)
        J[nr+1:end, i_cr] =  imag(e)

        # ∂/∂im(c_k) = im*e
        J[1:nr,     i_ci] = -imag(e)
        J[nr+1:end, i_ci] =  real(e)

        # ∂/∂freq_k = c * im * r * e   (chain rule: ×1000 from freq scaling)
        d = im .* r .* ce
        J[1:nr,     i_freq] = real(d) * 1000
        J[nr+1:end, i_freq] = imag(d) * 1000

        # ∂/∂γ_k: chain rule factor = decay[k] * (1 - decay[k]/DECAY_MAX)
        d_factor = decay[k] * (1 - decay[k] / DECAY_MAX)
        d = -r .* ce .* d_factor
        J[1:nr,     i_decay] = real(d)
        J[nr+1:end, i_decay] = imag(d)
    end

    return J
end


In [ ]:
B=[1.0,100.0,0.2,-2.0,
   0.5,50.0,0.22,-5]
r= NMRlab.coords(fidData,1)
Plots.plot(r,
    real(oscillators2(r,B)))

In many cases, we do not want to search for peaks in the entire spectrum,
but are interested in a finite range of frequencies only. This can be achieved
conveniently by digital filtering. Here are two routines that compute sinc-shaped
band-reject and band-pass filters, respectively:

In [ ]:
import DSP

function BandRejectFilter(lf,hf,n)
   b = [ t == 0 ? -ComplexF64(hf-lf,0.0) : 1.0/(2pi*im*t)*(exp(2pi*im*lf*t)-exp(2pi*im*hf*t)) for t=-n:n ] # Generalized sinc: invert frequencies in the reject band
   b .*= 0.42 .- 0.5*cos.(pi/n*(0:2n)) .+ 0.08* cos.(2pi/n*(0:2n))  # Blackman window  
   b[n+1] += ComplexF64(1.0,0.0) # Dirac delta function: All frequencies pass
   return b
end

function BandPassFilter(lf,hf,n)
   b = -[ t == 0 ? -ComplexF64(hf-lf,0.0) : 1.0/(2pi*im*t)*(exp(2pi*im*lf*t)-exp(2pi*im*hf*t)) for t=-n:n ] # Generalized sinc: invert frequencies in the reject band
   b .*= 0.42 .- 0.5*cos.(pi/n*(0:2n)) .+ 0.08* cos.(2pi/n*(0:2n))  # Blackman window  
   return b
end

function myfilt(f, data::SpectData{T,N}) where {T,N}
    # Apply the filter to each column of the data
    filtered_data = DSP.filt(f,data.dat)
    return SpectData(filtered_data,NMRlab.coords(data))
end

For example, we may only be interested in the aromatic part of the spectrum:

In [ ]:
dt = step(NMRlab.coords(fidData,1))
SW = 1/dt
H2OReject = BandRejectFilter(-60.0/SW,250.0/SW,1000)
aromaticSelect = BandPassFilter(500.0/SW,2400/SW,1000)

fidDataF = myfilt(H2OReject,fidData)

Fits are not stable unless we have reliable initial guesses for the frequencies. The amplitudes, decay rates, and phases are more robust. We therefore need a tool to estimate the frequencies present in the signal. This can be done either by picking peaks in the spectrum computed by FFT, or by a subspace approach, such as the ESPRIT algorithm:

In [ ]:
using Statistics

function pick_peaks(S, freqs; min_sep=nothing, threshold=nothing)
    # Estimate noise floor from the lower quartile of the spectrum
    noise = quantile(S, 0.9)
    thresh = isnothing(threshold) ? 5*noise : threshold

    # Minimum separation: default to ~2% of the frequency range
    sep = isnothing(min_sep) ? round(Int, 0.0002*length(S)) : min_sep

    # Find local maxima above threshold
    peaks = [i for i in 2:length(S)-1
             if S[i] > S[i-1] && S[i] > S[i+1] && S[i] > thresh]

    # Non-maximum suppression: within any window of width sep, keep only the tallest
    keep = Int[]
    for i in peaks
        if all(S[i] >= S[j] for j in peaks if abs(i-j) < sep)
            push!(keep, i)
        end
    end

    return freqs[keep], S[keep]
end


In [ ]:
using LsqFit
using FFTW

spect = collect(fidDataF[2000:8000,1])
r     = NMRlab.coords(fidDataF, 1)[2000:8000]

S = abs.(fft(spect))
freqs = fftfreq(length(S),1/step(r))
# peak_freqs = freqs[sortperm(S,rev=true)[1:n_osc]]./1000 .* 2pi

@show peak_freqs, peak_amps = pick_peaks(S,freqs,threshold=500)


Plots.plot(fftfreq(length(r),1/step(r)), abs.(fft(spect)))
Plots.plot!(peak_freqs, peak_freqs.* 0.0 .+ 3e4, seriestype=:sticks,
    xaxis=:flip, marker=:circle,xlims=(-3000,3000))

In [ ]:
@show n_osc = length(peak_freqs)
B0    = 5*rand(4*n_osc)

B0[4:4:end] .= log(pi/(DECAY_MAX-pi))
B0[3:4:end] = peak_freqs./1000.0 .* 2pi

fit = curve_fit(
    (r, B) ->[real(oscillators2(r,B) .- spect); imag(oscillators2(r,B) .- spect)],
    (r, B) -> oscillators2_jacobian(r,B),
    r, zeros(2*length(r)), B0
)

println("Parameters: ", fit.param)
println("Std errors: ", stderror(fit))
println("Peak freqs: ", peak_freqs)

In [ ]:
Plots.plot(r,real.(oscillators2(r,fit.param)),label="fit")
Plots.plot!(r,real.(spect),opacity=0.2,label="experimental")
Plots.plot!(r,real.(spect.-oscillators2(r,fit.param)),label="residual")

In [ ]:
fit_ppm = 1000*fit.param[3:4:end]/(2pi*600).+4.78 
fit_abs = abs.(fit.param[1:4:end].+im*fit.param[2:4:end])
fit_dec = fit.param[4:4:end]
Plots.plot(fit_ppm,fit_abs,seriestype=:sticks,xaxis=:flip,xlims=(-0.1,9.1))

ESPRIT is a more systematic algorithm to estimate the frequencies present in the signal. It is based on a singular value decomposition of the Hankel matrix
constructed from the FID data:

In [ ]:
using LinearAlgebra
function esprit(x, K)
    N = length(x)
    L = N ÷ 2
    M = N - L + 1

    X  = [x[i+j-1] for i in 1:M, j in 1:L]
    svdX = svd(X)
    Us = svdX.U[:, 1:K]

    Us_up = Us[1:M-1, :]
    Us_dn = Us[2:M,   :]

    # Least-squares: Us_dn ≈ Us_up * Φ
    Phi = Us_up \ Us_dn
    ωs  = angle.(eigvals(Phi))   # frequencies in rad/sample

    return sort(ωs), svdX.S
end

function unique_frequency_indices(freqs, tol)
    idx = sortperm(freqs)
    sorted = freqs[idx]

    keep = Int[]
    group_start = 1

    for i in 2:length(sorted)
        if sorted[i] - sorted[group_start] > tol
            # close group: return index of element nearest to group centre
            group_centre = (sorted[group_start] + sorted[i-1]) / 2
            best = argmin(abs(sorted[j] - group_centre) for j in group_start:i-1)
            push!(keep, idx[group_start + best - 1])
            group_start = i
        end
    end

    # close final group
    group_centre = (sorted[group_start] + sorted[end]) / 2
    best = argmin(abs(sorted[j] - group_centre) for j in group_start:length(sorted))
    push!(keep, idx[group_start + best - 1])

    return keep
end


In [ ]:
spect = collect(fidDataF[2000:8000,1])
peak_freqs, S = esprit(spect,200)./dt./(2pi)

Plots.plot(fftfreq(length(r),1/step(r)), abs.(fft(spect)))
Plots.plot!(peak_freqs, peak_freqs.* 0.0 .+ 25000, seriestype=:sticks,
    xaxis=:flip, marker=:circle, xlims=(-3000,2500))

In [ ]:
@show uidx = unique_frequency_indices(peak_freqs, 2.0)

Plots.plot(fftfreq(length(r),1/step(r)), abs.(fft(spect)))
Plots.plot!(peak_freqs[uidx], peak_freqs[uidx].* 0.0 .+ 25000, seriestype=:sticks,
    xaxis=:flip, marker=:circle, xlims=(-3000,2500))

In [ ]:
@show n_osc = length(uidx)
B0    = 0*rand(4*n_osc)
B0[1:4:end] = 5*rand(n_osc)   # real part of amplitude
B0[2:4:end] = 5*rand(n_osc)   # imag part of amplitude
B0[3:4:end] = peak_freqs[uidx]./1000.0 .* 2pi
B0[4:4:end] .= log(pi/(DECAY_MAX-pi)) # sigmoidal scaling of the parameter; start with pi/s decay rate

fit = curve_fit(
    (r, B) ->[real(oscillators2(r,B) .- spect); imag(oscillators2(r,B) .- spect)],
    (r, B) -> oscillators2_jacobian(r,B),
    r, zeros(2*length(r)), B0
)

println("Parameters: ", fit.param)
println("Std errors: ", stderror(fit))
println("Peak freqs: ", peak_freqs)

In [ ]:
Plots.plot(peak_freqs[uidx],fit.param[3:4:end].*1000 ./ (2pi), seriestype=:scatter)
Plots.plot!(peak_freqs[uidx],DECAY_MAX./(1 .+ exp.(-fit.param[4:4:end])),seriestype=:scatter,marker=:circle)

In [ ]:
Plots.plot(r,real.(oscillators2(r,fit.param)),label="fit")
Plots.plot!(r,real.(spect),opacity=0.2,label="experimental")
Plots.plot!(r,real.(spect.-oscillators2(r,fit.param)),label="residual")

In [ ]:
Plots.plot(fftfreq(length(r),1/step(r))/600.0.+4.78,abs.(fft(spect.-oscillators2(r,fit.param))),xaxis=:flip)

In [ ]:
fit_ppm = 1000*fit.param[3:4:end]/(2pi*600).+4.78 
fit_abs = abs.(fit.param[1:4:end].+im*fit.param[2:4:end])
fit_dec = fit.param[4:4:end]
Plots.plot(fit_ppm,fit_abs,seriestype=:sticks,xaxis=:flip,xlims=(-0.1,9.1),xlabel="ppm", ylabel="Amplitude")
Plots.plot!(fftfreq(length(r),1/step(r))/600.0.+4.78, -abs.(fft(spect))/100,ylims=(-50,50))
Plots.plot!(fftfreq(length(r),1/step(r))/600.0.+4.78, -abs.(fft(oscillators2(r,fit.param)))/100 .- 10 ,ylims=(-50,50))

In [ ]:
Plots.plot(fftfreq(length(r),1/step(r))/600.0.+4.78, abs.(fft(spect))/100,ylims=(-50,50),label="experimental",xaxis=:flip)
Plots.plot!(fftfreq(length(r),1/step(r))/600.0.+4.78, abs.(fft(oscillators2(r,fit.param)))/100 .+ 10 ,ylims=(-1,50),xlims=(3,4.1),label="fitted")

In [ ]:
Plots.plot((S[1:100]),seriestype=:scatter)

In [ ]:
Plots.plot(fit_ppm,pi./exp.(fit_dec),seriestype=:sticks,marker=:circle,xaxis=:flip,xlims=(-0.1,9.1),
    xlabel="ppm", ylabel="T2 (s)",ylims=(0,3.5))

In [ ]:
# log evidence ≈ log L_max - (K_params/2)*log(2π) + (1/2)*log|det(J'J)|
function laplace_log_evidence(fit)
    K=length(fit.param)
    n   = length(fit.resid)
    σ²  = sum(fit.resid.^2) / n
    JtJ = fit.jacobian' * fit.jacobian
    return -n/2 * log(σ²) + 1/2 * logabsdet(JtJ)[1] - 4K/2 * log(n)
end


In [ ]:
function fit_fid(fid::AbstractVector{<:Complex}, r::StepRangeLen, K::Int)
    dt = step(r)

    # Initial frequency estimates via ESPRIT
    ωs_per_sample, _ = esprit(fid, K)
    freq_hz = ωs_per_sample ./ dt ./ (2π)

    # B layout: [re(c), im(c), freq_param, γ, ...]
    # oscillators2 uses freq = 1000*B[3], so freq_param = freq_hz/1000*2π
    B0 = zeros(4K)
    B0[1:4:end] .= 1.0
    B0[2:4:end] .= -0.5
    B0[3:4:end]  = freq_hz ./ 1000.0 .* 2π
    B0[4:4:end] .= -5.0          # γ=0 → decay=1 s⁻¹

    fit = curve_fit(
        (r, B) -> [real(oscillators2(r, B) .- fid); imag(oscillators2(r, B) .- fid)],
        (r, B) -> oscillators2_jacobian(r, B),
        r, zeros(2*length(r)), B0
    )

    B          = fit.param
    freq_hz_fit = 1000.0 .* B[3:4:end] ./ (2π)
    amplitude  = abs.(B[1:4:end] .+ im .* B[2:4:end])
    phase_rad  = angle.(B[1:4:end] .+ im .* B[2:4:end])
    decay_rate = exp.(B[4:4:end])   # s⁻¹
    T2         = π ./ decay_rate    # s

    idx = sortperm(freq_hz_fit)

    return Dict(
        "n_oscillators" => K,
        "residual"      => sum(fit.resid.^2),
        "log_evidence"  => laplace_log_evidence(fit),
        "resonances"    => [
            Dict(
                "frequency_hz"  => freq_hz_fit[i],
                "amplitude"     => amplitude[i],
                "phase_rad"     => phase_rad[i],
                "decay_rate_hz" => decay_rate[i] / (2π),
                "T2_s"          => T2[i]
            )
            for i in idx
        ]
    )
end

function fit_fid(F::SpectData,K::Integer)
    return fit_fid(F.dat,NMRlab.coords(F,1),K)
end


In [ ]:
laplace_log_evidence(fit)
J=fit.jacobian
logabsdet(J'*J)

In [ ]:
rep=fit_fid(fidDataF[2000:8000,1],10)

In [ ]:
x = [x["frequency_hz"] for x in rep["resonances"]]
y = [x["decay_rate_hz"] for x in rep["resonances"]]
Plots.plot(x,y,seriestype=:line,xlims=(-2000,2000),ylims=(0,10))

In [ ]:
function fit_fid_fixed_freq(fid::AbstractVector{<:Complex}, r::StepRangeLen, freq_hz::Vector)
    K  = length(freq_hz)
    dt = step(r)

    # Fixed frequency params (same scaling as oscillators2: freq = 1000*B[3])
    freq_param = freq_hz ./ 1000.0 .* 2π

    # Model over reduced B: [cr₁, ci₁, γ₁, cr₂, ci₂, γ₂, ...]
    function model(r, B)
        pred = sum(1:K) do k
            c     = B[3k-2] + im*B[3k-1]
            decay = DECAY_MAX / (1 + exp(-B[3k]))
            c .* exp.((-decay + im*freq_param[k]) .* r)
        end
        return [real(pred .- fid); imag(pred .- fid)]
    end

    function jacobian(r, B)
        nr = length(r)
        J  = zeros(2nr, 3K)
        for k in 1:K
            c     = B[3k-2] + im*B[3k-1]
            decay = DECAY_MAX / (1 + exp(-B[3k]))
            e     = exp.((-decay + im*freq_param[k]) .* r)
            ce    = c .* e

            # ∂/∂cr_k
            J[1:nr,     3k-2] =  real(e);  J[nr+1:end, 3k-2] = imag(e)
            # ∂/∂ci_k
            J[1:nr,     3k-1] = -imag(e);  J[nr+1:end, 3k-1] = real(e)
            # ∂/∂γ_k
            d_factor = decay * (1 - decay / DECAY_MAX)
            d = -r .* ce .* d_factor
            J[1:nr,     3k]   =  real(d);  J[nr+1:end, 3k]   = imag(d)
        end
        return J
    end

    B0 = zeros(3K)
    B0[1:3:end] .= 1.0   # initial cr
    # γ=0 → decay = DECAY_MAX/2; adjust if needed

    fit = curve_fit(model, jacobian, r, zeros(2length(r)), B0)

    B          = fit.param
    amplitude  = abs.(B[1:3:end] .+ im .* B[2:3:end])
    phase_rad  = angle.(B[1:3:end] .+ im .* B[2:3:end])
    decay_rate = DECAY_MAX ./ (1 .+ exp.(-B[3:3:end]))

    return Dict(
        "n_oscillators" => K,
        "residual"      => sum(fit.resid.^2),
        "log_evidence"  => laplace_log_evidence(fit),
        "resonances"    => [
            Dict("frequency_hz"  => freq_hz[k],
                 "amplitude"     => amplitude[k],
                 "phase_rad"     => phase_rad[k],
                 "decay_rate_hz" => decay_rate[k] / (2π),
                 "T2_s"          => π / decay_rate[k])
            for k in sortperm(freq_hz)
        ]
    )
end


In [ ]:
freq_hz, _ = esprit(spect,K)

# rep=fit_fid_fixed_freq(fidDataF[2000:8000,1].dat,NMRlab.coords(fidDataF[2000:8000],1),peak_freqs[uidx])